In [1]:
import numpy as np 
import pandas as pd 


/home/lcmj2803/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/lcmj2803/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

In [5]:
data = np.load("psk_dataset_multiT_multiSNR.npz")

In [7]:
X = data['X'] 
Y = data['order']
snr_db = data['snr_db']
T_used = data['T']

In [9]:
N, T_max, _ = X.shape

print('X shape:', X.shape)
print('Y shape:', Y.shape)
print('SNR shape:', snr_db.shape)
print('T_used shape:', T_used.shape)

X shape: (72000, 2048, 2)
Y shape: (72000,)
SNR shape: (72000,)
T_used shape: (72000,)


In [10]:
def iq_row_to_complex(X_row:np.ndarray, T:int)->np.ndarray:
    I = X_row[:T, 0]
    Q = X_row[:T, 1]
    return I + 1j*Q

In [11]:
def normalize_power(x:np.ndarray, eps:float = 1e-12)->np.ndarray:
    p = np.mean(np.abs(x)**2)
    if p < eps:
        return x
    return x / np.sqrt(p)    

In [12]:
def compute_constellation_features(xc:np.ndarray)->np.ndarray:
    xr = np.real(xc)
    ximg = np.imag(xc)
    
    #medias y varianzas
    mu_r = np.mean(xr)
    mu_i = np.mean(ximg)
    var_r = np.var(xr)
    var_i = np.var(ximg)
    
    #correlacion IQ
    cov_ri = np.mean( (xr - mu_r) * (ximg - mu_i) )
    std_r = np.sqrt(var_r + 1e-12)
    srd_i = np.sqrt(var_i + 1e-12)
    corr_ri = cov_ri / (std_r * srd_i + 1e-12)
    
    #fracción de puntos en cada cuadrante
    N = len(xc)
    q1 = np.mean((xr > 0) & (ximg > 0))
    q2 = np.mean((xr < 0) & (ximg > 0))
    q3 = np.mean((xr < 0) & (ximg < 0))
    q4 = np.mean((xr > 0) & (ximg < 0))
    
    return np.array([
        mu_r, mu_i,
        var_r, var_i,
        corr_ri,
        q1,q2,q3,q4], dtype=np.float32)

In [13]:
def compute_spectrum_features(xc: np.ndarray, NFFT: int =256, K_spec:int =128)->np.ndarray:
    x_pad = xc
    if len(x_pad) < NFFT:
        x_pad = np.pad(x_pad, (0, NFFT - len(x_pad)))
    else :
        x_pad = x_pad[:NFFT]
        
    Xf = np.fft.fft(x_pad, n=NFFT)
    mag = np.abs(Xf)
    
    #Nos quedamos en la primeras K_spec componentes
    K=min(K_spec, len(mag))
    feat = mag[:K]
    
    #Escala log 
    feat = np.log(feat + 1e-8)
    
    return feat.astype(np.float32)

In [14]:
def compute_cumulant_features(xc:np.ndarray)->np.ndarray:
    x = xc
    N = len(x)
    if N < 10:
        return np.zeros(10, dtype=np.float32)
    
    mu = np.mean(x)
    x_c = x - mu
    
    m20 = np.mean(x_c**2)
    m21 = np.mean(np.abs(x_c)**2)
    m40 = np.mean(x_c**4)
    m41 = np.mean((np.abs(x_c)**2) * (x_c**2))
    m42 = np.mean(np.abs(x_c)**4)
    
    C20 = m20
    C21 = m21 - np.abs(mu)**2
    C40 = m40 - 3*m20**2
    C41 = m41 - 3*m21*m20
    C42 = m42 - np.abs(m20)**2 - 2*m21**2
    
    feats = np.array([
        np.real(C20), np.imag(C20),
        np.real(C21), np.imag(C21),
        np.real(C40), np.imag(C40),
        np.real(C41), np.imag(C41),
        np.real(C42), np.imag(C42),
    ], dtype=np.float32)
    
    return feats
    
    

In [15]:
def fuse_features(f_const:np.ndarray,
                  f_spec:np.ndarray,
                  f_cum:np.ndarray)->np.ndarray:
    return np.concatenate([f_const, f_spec, f_cum], axis=-1)

In [16]:
N, T_max, _ = X.shape
NFFT = 256
K_spec = 128

constel_list = []
spec_list = []
cum_list = []

for i in range (N):
    T = int(T_used[i])
    xc = iq_row_to_complex(X[i], T)
    xc = normalize_power(xc)
    
    f_const = compute_constellation_features(xc)
    f_spec = compute_spectrum_features(xc, NFFT=NFFT, K_spec=K_spec)
    f_cum = compute_cumulant_features(xc)
    
    constel_list.append(f_const)
    spec_list.append(f_spec)
    cum_list.append(f_cum)

In [18]:
X_constel = np.stack(constel_list, axis=0)
X_spec = np.stack(spec_list, axis=0)
X_cum = np.stack(cum_list, axis=0)
X_fusion = np.concatenate([X_constel, X_spec, X_cum], axis=1)

print('X_constel shape:', X_constel.shape)
print('X_spec shape:', X_spec.shape)
print('X_cum shape:', X_cum.shape)
print('X_fusion shape:', X_fusion.shape)

X_constel shape: (72000, 9)
X_spec shape: (72000, 128)
X_cum shape: (72000, 10)
X_fusion shape: (72000, 147)


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X_fusion, Y, test_size=0.2, random_state=42, stratify=Y) 

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, multi_class='auto',n_jobs=-1))
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Reporte de Clasificación:",classification_report(y_test, y_pred))

print("Matriz de Confusión:",confusion_matrix(y_test, y_pred))

/home/lcmj2803/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/lcmj2803/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/lcmj2803/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


Reporte de Clasificación:               precision    recall  f1-score   support

           2       1.00      1.00      1.00      4800
           4       0.97      0.98      0.98      4800
           8       0.98      0.97      0.98      4800

    accuracy                           0.98     14400
   macro avg       0.98      0.98      0.98     14400
weighted avg       0.98      0.98      0.98     14400

Matriz de Confusión: [[4800    0    0]
 [   0 4702   98]
 [   0  123 4677]]


In [ ]:
np.savez("psk_classification_results.npz",
         X_fusion=X_fusion,
         order=Y,
         snr_db=snr_db,
         T = T_used
)

print("features and results saved to psk_classification_results.npz")